<a href="https://colab.research.google.com/github/AufanT/AufanT-BigData26_B_2411532011_AufanTaufiqurrahman/blob/main/Praktikum_5/BD_P05_2411532011_AufanTaufiqurrahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

In [2]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = ("https://raw.githubusercontent.com/ibamibrahim/"
            "dataset-judul-berita-indonesia/master/detik_news_title.csv")
if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")
print("Ukuran berkas :", round(os.path.getsize("detik_news_title.csv") / 1e6, 1), "MB")
print("Baris, kolom  :", df_sumber.shape)
print("Kolom         :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom  : (91017, 4)
Kolom         : ['date', 'url', 'title', 'category']


In [3]:
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100               # judul per halaman

def jalankan_portal(df, port=8000):
    """Portal berita lokal (siap pakai): menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):
        def log_message(self, *args):            # matikan log
            pass
        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        def do_GET(self):
            u = urllib.parse.urlparse(self.path)
            if u.path == "/robots.txt":
                self._kirim(200, "User-agent: *\nDisallow: /arsip-privat/\n", "text/plain")
            elif u.path == "/arsip-privat/":
                self._kirim(200, "<p>Halaman ini tidak boleh di-crawl.</p>")
            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])
                if not 1 <= hal <= n_hal:
                    return self._kirim(404, "<p>Halaman tidak ada</p>")
                potong = baris[(hal - 1) * HALAMAN_UKURAN: hal * HALAMAN_UKURAN]
                isi = []
                for r in potong:
                    iso = pd.to_datetime(r["date"], format="%m/%d/%Y").strftime("%Y-%m-%d")
                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a href="{html.escape(r["url"])}">{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">{html.escape(r["date"])}</time>'
                        '</article>')
                self._kirim(200, f"<html><body><h1>Portal Contoh</h1>{''.join(isi)}"
                                 f'<nav data-total="{n_hal}"></nav></body></html>')
            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, n_hal

server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"
print("Portal lokal aktif di", BASE, "| jumlah halaman:", N_HALAMAN)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


In [4]:
import requests, urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"

def baca_robots(base):
    r = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=10)
    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())
    return rp

robots = baca_robots(BASE)
for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


In [5]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict (judul, url, kategori, tanggal)."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []
    for a in sup.select("article.item"):                      # BAGIAN KUNCI
        link = a.select_one("h2.judul a")
        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one("span.kategori").get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })
    return hasil

def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Siap pakai: ambil halaman satu per satu, patuhi robots.txt, beri jeda."""
    rp = baca_robots(base)
    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua
    for h in halaman:
        url = f"{base}/berita?hal={h}"
        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue
        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                time.sleep(0.5 * (percobaan + 1))
        else:
            print("Gagal setelah 3 percobaan:", url)
            continue
        yield from parse_halaman(r.text)
        time.sleep(delay)

contoh = list(crawl_iter(BASE, range(1, 4)))
print("Judul dari 3 halaman:", len(contoh))
for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


In [6]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/")

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita (XML) menjadi daftar dict (judul, url, kategori, id, tanggal, kata_kunci)."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []
    for u in sup.find_all("url"):
        loc, judul, waktu = u.find("loc"), u.find("title"), u.find("publication_date")   # BAGIAN KUNCI
        if loc is None or judul is None:
            continue
        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")
        hasil.append({
            "judul": re.sub(r"\s+", " ", judul.get_text(strip=True)),
            "url": url,
            "kategori": m.group(1) if m else urlparse(url).netloc.split(".")[0],  # BAGIAN KUNCI: news, inet, sport, ...
            "id": int(m.group(2)) if m else None,
            "tanggal": waktu.get_text(strip=True)[:10] if waktu else None,        # BAGIAN KUNCI: ambil tanggal saja
            "kata_kunci": kunci.get_text(strip=True) if kunci else "",
        })
    return hasil

In [7]:
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication><news:name>detikNews</news:name><news:language>id</news:language></news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({k: x[k] for k in ("judul", "kategori", "id", "tanggal")})
print("kata kunci entri pertama:", parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"])
print("Sama bila masukan berupa bytes:", parse_sitemap_news(CONTOH_XML.encode("utf-8")) == parse_sitemap_news(CONTOH_XML))

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


In [8]:
import urllib.robotparser

EMAIL_KONTAK = "aufantaufiq08905@gmail.com"
UA_DETIK = f"BigData/1.0 (praktikum pembelajaran; kontak: {EMAIL_KONTAK})"
DELAY = 2.0                 # detik antar-request. Jangan dikurangi.
BATAS_PER_BAGIAN = 7        # judul yang diambil dari tiap sitemap
BATAS_TOTAL = 20            # batas seluruh crawling
SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}
def boleh_diambil(url, ua=UA_DETIK):
    """robots.txt berlaku per host: news.detik.com dan inet.detik.com punya berkas sendiri."""
    host = urlparse(url).netloc
    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp
    return _robots[host].can_fetch(ua, url)

def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)
    r = sesi.get(url, timeout=20)
    if r.status_code in (403, 429):
        raise RuntimeError(f"Server membalas {r.status_code}. Berhenti, tunggu, dan kurangi laju.")
    r.raise_for_status()
    time.sleep(delay)
    return r.content            # bytes: pengkodean dibaca dari deklarasi XML

def crawl_detik(sitemap=SITEMAP_DIPILIH):
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK
    hasil = []
    for sm in sitemap:
        if len(hasil) >= BATAS_TOTAL:
            break
        hasil += parse_sitemap_news(ambil_xml(sm, sesi))[:BATAS_PER_BAGIAN]
    return hasil[:BATAS_TOTAL]

In [9]:
try:
    sampel = crawl_detik()
except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)
    with open("contoh_crawl/detik_sitemap.jsonl", "w", encoding="utf-8") as f:
        for x in sampel:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    cek = pd.DataFrame(sampel)
    print("Judul terkumpul   :", len(cek))
    print("Judul kosong      :", (cek["judul"].str.strip() == "").sum())
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())
    print(cek[["judul", "kategori", "tanggal"]].head(3).to_string(index=False))

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                                      judul kategori    tanggal
 Polisi Tangkap Pengedar Obat Keras di Bogor, 282.500 Butir Tramadol Disita     news 2026-11-13
WNI Tentara Bayaran Rusia Dijanjikan Bonus Jika Bisa Tembak Tentara Ukraina     news 2026-11-13
          Di COP31 Turki, DPD RI Tawarkan 3 Usulan Kebijakan Komitmen Iklim     news 2026-11-13


In [10]:
RAW = Path("lapisan_mentah"); RAW.mkdir(exist_ok=True)
for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()
statistik = {"max_antrean": 0, "masuk": 0, "keluar": 0, "file": 0}

def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)                       # menunggu bila channel penuh
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(statistik["max_antrean"], channel.qsize())
    channel.put(STOP)

def sink(ukuran_batch=5000):
    batch, nomor = [], 0
    def tulis():
        nonlocal batch, nomor
        nomor += 1
        with open(RAW / f"batch_{nomor:04d}.jsonl", "w", encoding="utf-8") as f:
            for ev in batch:
                f.write(json.dumps(ev, ensure_ascii=False) + "\n")
        statistik["file"] = nomor
        batch = []
    while True:
        ev = channel.get()
        if ev is STOP:
            break
        batch.append(ev); statistik["keluar"] += 1
        if len(batch) >= ukuran_batch:
            tulis()
    if batch:
        tulis()

t0 = time.perf_counter()
t_src = threading.Thread(target=source); t_snk = threading.Thread(target=sink)
t_src.start(); t_snk.start(); t_src.join(); t_snk.join()
waktu_crawl = time.perf_counter() - t0
print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink  :", f"{statistik['keluar']:,}")
print("File batch          :", statistik["file"], "->", sorted(p.name for p in RAW.glob('*.jsonl'))[:3], "...")
print("Antrean maks <= 2000:", statistik["max_antrean"] <= 2000, f"(maks teramati: {statistik['max_antrean']})")
print("Total ukuran mentah :", round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1), "MB")
print("Waktu crawl total   :", round(waktu_crawl, 1), "detik untuk", N_HALAMAN, "halaman")

Event masuk channel : 91,017
Event ditulis sink  : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True (maks teramati: 200)
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 34.9 detik untuk 911 halaman


In [11]:
mentah = pd.concat([pd.read_json(p, lines=True) for p in sorted(RAW.glob("*.jsonl"))],
                   ignore_index=True)
print("Baris mentah:", f"{len(mentah):,}")

KATEGORI_DIKENAL = {"finance", "food", "health", "hot", "inet", "news", "oto", "sport", "travel"}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()
    return {
        "judul tidak kosong"   : d["judul"].fillna("").str.strip() != "",
        "judul 3-30 kata"      : jml_kata.between(3, 30),                                         # BAGIAN KUNCI
        "kategori dikenal"     : d["kategori"].isin(KATEGORI_DIKENAL),                            # BAGIAN KUNCI
        "tanggal valid"        : pd.to_datetime(d["tanggal"], format="%Y-%m-%d", errors="coerce").notna(),
        "url artikel valid"    : d["url"].fillna("").str.contains(r"/d-\d+", regex=True),         # BAGIAN KUNCI
        "judul tidak duplikat" : ~d["judul"].duplicated(keep=False),
    }

for nama, lulus in aturan_kualitas(mentah).items():
    print(f"{nama:<24} gagal: {(~lulus).sum():>6,} ({(~lulus).mean()*100:5.2f}%)")

Baris mentah: 91,017
judul tidak kosong       gagal:      0 ( 0.00%)
judul 3-30 kata          gagal:     78 ( 0.09%)
kategori dikenal         gagal:      0 ( 0.00%)
tanggal valid            gagal:      0 ( 0.00%)
url artikel valid        gagal:  5,640 ( 6.20%)
judul tidak duplikat     gagal:    235 ( 0.26%)


In [12]:
# Bukti temuan: baris yang gagal "url artikel valid" berasal dari kategori apa?
gagal_url = ~aturan_kualitas(mentah)["url artikel valid"]
print("Kategori baris dengan url rusak:", mentah.loc[gagal_url, "kategori"].value_counts().to_dict())
print("Contoh nilai url rusak         :", mentah.loc[gagal_url, "url"].unique()[:3])

Kategori baris dengan url rusak: {'inet': 5640}
Contoh nilai url rusak         : ['#']


In [13]:
bersih = mentah.copy()
bersih["judul"] = bersih["judul"].str.replace(r"\s+", " ", regex=True).str.strip()
bersih = bersih.drop_duplicates(subset="judul", keep="first")
bersih = bersih[bersih["judul"].str.split().str.len().between(3, 30)].copy()
bersih["tanggal"] = pd.to_datetime(bersih["tanggal"], format="%Y-%m-%d")
bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")
bersih["id"] = bersih["url"].str.extract(r"/d-(\d+)")[0].astype("Int64")   # kosong bila url rusak
bersih["url"] = bersih["url"].where(bersih["url"].str.contains(r"/d-\d+", regex=True))
print("Baris setelah dibersihkan:", f"{len(bersih):,}", "| dibuang:", f"{len(mentah) - len(bersih):,}")

Baris setelah dibersihkan: 90,818 | dibuang: 199


In [14]:
import shutil, pyarrow.dataset as ds

def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)
    data.to_parquet(folder, partition_cols=kolom_partisi, index=False)
    n = len(list(Path(folder).rglob("*.parquet")))
    ukuran = sum(p.stat().st_size for p in Path(folder).rglob("*.parquet")) / 1e6
    return n, ukuran

hasil_partisi = {}
for kolom in (["kategori"], ["bulan"], ["kategori", "bulan"]):
    n, mb = simpan_terpartisi(bersih, kolom, "uji_partisi")
    hasil_partisi[str(kolom)] = (n, round(mb, 1))
    print(f"partisi {str(kolom):<22} -> {n:>3} file, {mb:5.1f} MB")
shutil.rmtree("uji_partisi")

TERKURASI = "lapisan_terkurasi"
n, mb = simpan_terpartisi(bersih, ["kategori"], TERKURASI)          # BAGIAN KUNCI: partisi kategori
print("\nDisimpan ke", TERKURASI, "dengan partisi kategori:", n, "file,", round(mb, 1), "MB")

dataset = ds.dataset(TERKURASI, format="parquet", partitioning="hive")
semua = list(dataset.get_fragments())
hanya = list(dataset.get_fragments(filter=ds.field("kategori") == "finance"))
print("Partition pruning: baca", len(hanya), "dari", len(semua), "file untuk kategori = finance")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB

Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


In [15]:
import duckdb
con = duckdb.connect()
con.sql(f"CREATE VIEW berita AS SELECT * FROM read_parquet('{TERKURASI}/*/*.parquet', hive_partitioning=true)")

per_kat_df = con.sql("SELECT kategori, COUNT(*) AS jumlah FROM berita GROUP BY kategori ORDER BY jumlah DESC").df()
print(per_kat_df.to_string(index=False))
print()
corona_df = con.sql("""
    SELECT bulan, COUNT(*) AS judul_corona
    FROM berita WHERE lower(judul) LIKE '%corona%'
    GROUP BY bulan ORDER BY bulan""").df()
print(corona_df.to_string(index=False))

dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(bersih.iloc[0]["tanggal"].date()),
    "kata": re.findall(r"[a-z]+", bersih.iloc[0]["judul"].lower()),
}
print()
print(json.dumps(dokumen, ensure_ascii=False, indent=2))

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


In [16]:
corona_kat_df = con.sql("""
    SELECT kategori,
           COUNT(*) FILTER (WHERE lower(judul) LIKE '%corona%') AS judul_corona,
           COUNT(*) AS total,
           ROUND(100.0 * COUNT(*) FILTER (WHERE lower(judul) LIKE '%corona%') / COUNT(*), 1) AS persen
    FROM berita GROUP BY kategori ORDER BY persen DESC""").df()
print(corona_kat_df.to_string(index=False))

kategori  judul_corona  total  persen
  health          2591   4914    52.7
    news          7532  32321    23.3
 finance          2473  14141    17.5
  travel          1025   6453    15.9
    inet           847   5577    15.2
     oto           577   4375    13.2
     hot          1663  16290    10.2
   sport           228   2433     9.4
    food           310   4314     7.2


In [17]:
kasir = [
    ["nasi goreng,1", "es teh,2", "mie ayam,1", "es teh,1"],      # kasir A
    ["nasi goreng,2", "es teh,1", "es teh,3", "mie ayam,1"],      # kasir B
    ["mie ayam,2", "nasi goreng,1", "es teh,1", "nasi goreng,1"], # kasir C
]

def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))

def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

# 1) MAP: setiap kasir memproses datanya sendiri
hasil_map = [[map_fn(b) for b in k] for k in kasir]
for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)

# 2) SHUFFLE: kumpulkan nilai berdasarkan key
shuffle = {}
for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)
print("\nshuffle:", shuffle)

# 3) REDUCE: gabungkan nilai per key
hasil = dict(reduce_fn(m, v) for m, v in shuffle.items())
print("\nreduce:", hasil, "| total porsi:", sum(hasil.values()))

# Combiner: jumlahkan dulu di setiap kasir sebelum dikirim
def combiner(pasangan):
    c = collections.Counter()
    for menu, jumlah in pasangan:
        c[menu] += jumlah
    return list(c.items())

setelah = [combiner(h) for h in hasil_map]
print("\nPasangan dikirim tanpa combiner:", sum(len(h) for h in hasil_map))
print("Pasangan dikirim dengan combiner:", sum(len(h) for h in setelah))

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17

Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


In [18]:
STOPWORD = set("""yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn""".split())

def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

# Input split = satu file partisi Parquet (satu kategori)
splits = sorted(Path(TERKURASI).rglob("*.parquet"))
print("Jumlah input split:", len(splits))
for p in splits:
    n = len(pd.read_parquet(p, columns=["judul"]))
    print(f"  {p.parent.name:<16} {n:>6,} judul")

def mapper_split(path):
    """Satu mapper memproses satu split dan mengembalikan pasangan (kata, 1)."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    return [(kata, 1) for t in judul for kata in tokenisasi(t)]

hasil_map = [mapper_split(p) for p in splits]
total_pasangan = sum(len(h) for h in hasil_map)

shuffle = collections.defaultdict(list)
for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)
hasil_semua = {k: sum(v) for k, v in shuffle.items()}

print("\nPasangan dari semua mapper:", f"{total_pasangan:,}")
print("Kata unik (key setelah shuffle):", f"{len(hasil_semua):,}")
print("\n10 kata teratas (semua kata):")
for k, n in collections.Counter(hasil_semua).most_common(10):
    print(f"  {k:<10} {n:>7,}")
top = [(k, n) for k, n in collections.Counter(hasil_semua).most_common() if k not in STOPWORD]
print("\n10 kata teratas (tanpa stopword):")
for k, n in top[:10]:
    print(f"  {k:<10} {n:>7,}")

# Cek silang dengan cara biasa (tanpa MapReduce)
cek = collections.Counter(kata for t in bersih["judul"] for kata in tokenisasi(t))
print("\nSama dengan Counter biasa:", dict(cek) == hasil_semua)

Jumlah input split: 9
  kategori=finance 14,141 judul
  kategori=food     4,314 judul
  kategori=health   4,914 judul
  kategori=hot     16,290 judul
  kategori=inet     5,577 judul
  kategori=news    32,321 judul
  kategori=oto      4,375 judul
  kategori=sport    2,433 judul
  kategori=travel   6,453 judul

Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
  di          27,390
  corona      17,239
  ini         10,124
  yang         8,668
  dan          6,785
  ke           5,592
  tak          5,096
  dari         4,697
  jadi         4,486
  untuk        3,723

10 kata teratas (tanpa stopword):
  corona      17,239
  rp           3,154
  virus        3,075
  kasus        2,743
  polisi       2,732
  positif      2,682
  indonesia    2,504
  warga        2,495
  rumah        2,465
  jokowi       2,317

Sama dengan Counter biasa: True


In [19]:
# Peringkat "rp", "ri", "the" setelah stopword dibuang (bahan Analisis Hasil no. 3)
peringkat = {k: i + 1 for i, (k, _) in enumerate(top)}
for k in ("rp", "ri", "the"):
    print(f"{k:<4} peringkat {peringkat.get(k)} | frekuensi {hasil_semua.get(k, 0):,}")

rp   peringkat 2 | frekuensi 3,154
ri   peringkat 12 | frekuensi 2,206
the  peringkat 102 | frekuensi 678


In [20]:
from multiprocessing import Pool

def mapper_combiner(path):
    """Mapper + combiner: hitung dulu di sisi mapper."""
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    c = collections.Counter()
    for t in judul:
        c.update(tokenisasi(t))
    return c

def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()
    if proses == 1:
        hasil = [fn(x) for x in daftar]
    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)
    return hasil, time.perf_counter() - t0

tabel_h10 = []
print("CPU tersedia:", os.cpu_count())
print(f"{'skenario':<28}{'waktu (s)':>10}{'pasangan dikirim':>20}")
for proses in (1, 2):
    waktu = []
    for _ in range(3):
        h, w = jalankan(mapper_split, splits, proses); waktu.append(w)
    print(f"{proses} proses, tanpa combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h):>20,}")
    tabel_h10.append((f"{proses} proses, tanpa combiner", round(sorted(waktu)[1], 3), sum(len(x) for x in h)))
    waktu = []
    for _ in range(3):
        h2, w = jalankan(mapper_combiner, splits, proses); waktu.append(w)
    print(f"{proses} proses, dengan combiner".ljust(28), f"{sorted(waktu)[1]:>10.3f}", f"{sum(len(x) for x in h2):>20,}")
    tabel_h10.append((f"{proses} proses, dengan combiner", round(sorted(waktu)[1], 3), sum(len(x) for x in h2)))

total = collections.Counter()
for c in h2:
    total.update(c)
print("\nHasil dengan combiner sama persis:", dict(total) == hasil_semua)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner          0.476              825,601
1 proses, dengan combiner         0.608               74,009
2 proses, tanpa combiner          0.721              825,601
2 proses, dengan combiner         0.380               74,009

Hasil dengan combiner sama persis: True


In [21]:
R = 4           # jumlah reducer

def partitioner(kata, r=R):
    return zlib.crc32(kata.encode()) % r

def beban(pasangan_per_mapper, r=R):
    b = [0] * r
    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1
    return b

def tampil(nama, b):
    tot = sum(b)
    print(f"{nama:<22}", " ".join(f"R{i}={x:>7,} ({x/tot*100:4.1f}%)" for i, x in enumerate(b)))

tampil("tanpa combiner", beban(hasil_map))
tampil("dengan combiner", beban([list(c.items()) for c in h2]))

print("\nBeban reducer terberat dibanding bagian yang adil (tanpa combiner):")
for r in (4, 16, 64):
    b = beban(hasil_map, r)
    print(f"  {r:>3} reducer: terberat {max(b)/sum(b)*100:5.2f}% | bagian adil {100/r:5.2f}%")

print("\nBeban satu key (tanpa combiner):")
tot = sum(hasil_semua.values())
for kata, n in collections.Counter(hasil_semua).most_common(5):
    print(f"  {kata:<8} {n:>7,} pasangan = {n/tot*100:4.1f}% dari total, di reducer R{partitioner(kata)}")

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)

Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
    4 reducer: terberat 29.17% | bagian adil 25.00%
   16 reducer: terberat  8.82% | bagian adil  6.25%
   64 reducer: terberat  4.28% | bagian adil  1.56%

Beban satu key (tanpa combiner):
  di        27,390 pasangan =  3.3% dari total, di reducer R0
  corona    17,239 pasangan =  2.1% dari total, di reducer R2
  ini       10,124 pasangan =  1.2% dari total, di reducer R0
  yang       8,668 pasangan =  1.0% dari total, di reducer R3
  dan        6,785 pasangan =  0.8% dari total, di reducer R3


In [22]:
Path("masukan.tsv").write_text(
    "\n".join(f"{k}\t{j}" for k, j in zip(bersih["kategori"], bersih["judul"])) + "\n",
    encoding="utf-8")

Path("mapper.py").write_text('''import sys, re
for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)
    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys
kunci_sekarang, total = None, 0
for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")
    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")
        kunci_sekarang, total = kunci, 0
    total += int(nilai)
if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')
print("Baris masukan:", sum(1 for _ in open("masukan.tsv", encoding="utf-8")))

Baris masukan: 90818


In [23]:
!head -n 3 masukan.tsv | python3 mapper.py

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1


In [24]:
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5


real	0m1.109s
user	0m1.195s
sys	0m0.056s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


In [25]:
hasil_pipe = {}
for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)
print("Hasil pipe sama dengan hasil H-9:", hasil_pipe == hasil_semua)

Hasil pipe sama dengan hasil H-9: True


In [26]:
!cat masukan.tsv | python3 mapper.py | python3 reducer.py > hasil_tanpa_sort.txt
tanpa_sort = collections.defaultdict(list)
for baris in open("hasil_tanpa_sort.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    tanpa_sort[kata].append(int(n))
print("Baris keluaran tanpa sort :", f"{sum(len(v) for v in tanpa_sort.values()):,}  (dengan sort: {len(hasil_pipe):,})")
print("Kata yang muncul >1 kali  :", f"{sum(1 for v in tanpa_sort.values() if len(v) > 1):,}")
print("Contoh 'corona' tanpa sort:", len(tanpa_sort["corona"]), "baris, nilai maks satu baris =", max(tanpa_sort["corona"]),
      "| seharusnya total", hasil_semua["corona"])

Baris keluaran tanpa sort : 821,949  (dengan sort: 32,275)
Kata yang muncul >1 kali  : 21,214
Contoh 'corona' tanpa sort: 17210 baris, nilai maks satu baris = 2 | seharusnya total 17239


In [40]:
%%bash
echo "== 1. Instal Java =="
sudo apt-get update -qq > /dev/null 2>&1
sudo apt-get install -y -qq openjdk-11-jdk-headless > apt.log 2>&1 \
  || sudo apt-get install -y -qq openjdk-17-jdk-headless >> apt.log 2>&1 \
  || { echo "GAGAL instal Java:"; tail -n 5 apt.log; exit 1; }
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))
java -version 2>&1 | head -1
echo "JAVA_HOME=$JAVA_HOME"

echo "== 2. Unduh Hadoop 3.3.6 (±700 MB) =="
if [ ! -d hadoop-3.3.6 ]; then
  wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz \
    || { echo "GAGAL unduh Hadoop"; exit 1; }
  tar -xzf hadoop-3.3.6.tar.gz && rm hadoop-3.3.6.tar.gz
fi
ls -d hadoop-3.3.6

echo "== 3. Jalankan Hadoop Streaming =="
export HADOOP_HOME=$PWD/hadoop-3.3.6
rm -rf keluaran_hadoop
$HADOOP_HOME/bin/hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar \
  -files mapper.py,reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
  -input masukan.tsv -output keluaran_hadoop > log_hadoop.txt 2>&1
echo "exit code: $?"
tail -n 25 log_hadoop.txt

echo "== 4. Bandingkan dengan hasil pipe =="
ls keluaran_hadoop
diff <(LC_ALL=C sort hasil_pipe.txt) keluaran_hadoop/part-00000 > /dev/null && echo "SAMA" || echo "BERBEDA"

== 1. Instal Java ==
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
== 2. Unduh Hadoop 3.3.6 (±700 MB) ==
hadoop-3.3.6
== 3. Jalankan Hadoop Streaming ==
exit code: 0
		Input split bytes=77
		Combine input records=0
		Combine output records=0
		Reduce input groups=32275
		Reduce shuffle bytes=8695931
		Reduce input records=825601
		Reduce output records=32275
		Spilled Records=1651202
		Shuffled Maps =1
		Failed Shuffles=0
		Merged Map outputs=1
		GC time elapsed (ms)=25
		Total committed heap usage (bytes)=432013312
	Shuffle Errors
		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=6078704
	File Output Format Counters 
		Bytes Written=334406
2026-11-13 15:47:57,500 INFO streaming.StreamJob: Output directory: keluaran_hadoop
== 4. Bandingkan dengan

In [28]:
def mapper_kategori(path, stopword=None):
    """Key gabungan (kategori, kata); combiner dipasang di mapper."""
    stopword = STOPWORD if stopword is None else stopword
    df_ = pd.read_parquet(path, columns=["judul"])
    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()
    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in stopword:
                c[(kategori, kata)] += 1
    return c

def top_per_kategori(gabung, n=5):
    per_kategori = collections.defaultdict(list)
    for (kat, kata), jml in gabung.items():
        per_kategori[kat].append((jml, kata))
    return {kat: [(k, j) for j, k in sorted(v, reverse=True)[:n]] for kat, v in sorted(per_kategori.items())}

gabung = collections.Counter()
for c in map(mapper_kategori, splits):
    gabung.update(c)

top5_awal = top_per_kategori(gabung)
for kat, daftar in top5_awal.items():
    print(f"{kat:<8}", ", ".join(f"{k} ({n:,})" for k, n in daftar))

import sqlite3
SIAP = Path("siap_pakai"); SIAP.mkdir(exist_ok=True)
baris = [(kat, kata, n) for (kat, kata), n in gabung.items()]
hasil_df = pd.DataFrame(baris, columns=["kategori", "kata", "jumlah"])
hasil_df.to_parquet(SIAP / "kata_per_kategori.parquet", index=False)

# "sqoop export": pindahkan hasil ke database relasional
db = sqlite3.connect(SIAP / "laporan.db")
hasil_df.to_sql("kata_per_kategori", db, if_exists="replace", index=False)
print("\nBaris diekspor:", db.execute("SELECT COUNT(*) FROM kata_per_kategori").fetchone()[0])
print(db.execute("""SELECT kata, jumlah FROM kata_per_kategori
                    WHERE kategori = 'health' ORDER BY jumlah DESC LIMIT 3""").fetchall())
db.close()

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


### Latihan 1 — Stopword domain
Lima kata tambahan yang mengganggu: `rp`, `ri`, `the`, `foto`, `galeri` (satuan mata uang, singkatan negara, kata bahasa Inggris pada judul film/lagu, dan label format konten galeri, bukan topik).

In [29]:
TAMBAHAN = {"rp", "ri", "the", "foto", "galeri"}
STOPWORD_DOMAIN = STOPWORD | TAMBAHAN

gabung_domain = collections.Counter()
for p in splits:
    gabung_domain.update(mapper_kategori(p, STOPWORD_DOMAIN))
top5_domain = top_per_kategori(gabung_domain)

for kat in top5_awal:
    lama = [k for k, _ in top5_awal[kat]]
    baru = [k for k, _ in top5_domain[kat]]
    masuk = [k for k in baru if k not in lama]
    print(f"{kat:<8} sebelum: {', '.join(lama)}")
    print(f"{'':<8} sesudah: {', '.join(baru)}   | masuk: {masuk or '-'}")

finance  sebelum: corona, rp, ri, harga, jokowi
         sesudah: corona, harga, jokowi, mau, ekonomi   | masuk: ['mau', 'ekonomi']
food     sebelum: makanan, makan, foto, galeri, enak
         sesudah: makanan, makan, enak, bikin, rumah   | masuk: ['bikin', 'rumah']
health   sebelum: corona, virus, pasien, indonesia, meninggal
         sesudah: corona, virus, pasien, indonesia, meninggal   | masuk: -
hot      sebelum: corona, anak, rumah, meninggal, the
         sesudah: corona, anak, rumah, meninggal, film   | masuk: ['film']
inet     sebelum: corona, virus, ponsel, bikin, indonesia
         sesudah: corona, virus, ponsel, bikin, indonesia   | masuk: -
news     sebelum: corona, polisi, kasus, warga, positif
         sesudah: corona, polisi, kasus, warga, positif   | masuk: -
oto      sebelum: mobil, corona, motor, rp, indonesia
         sesudah: mobil, corona, motor, indonesia, honda   | masuk: ['honda']
sport    sebelum: corona, indonesia, motogp, masters, olimpiade
         sesudah

### Latihan 2 — Key (bulan, kata)

In [30]:
def mapper_bulan(path):
    df_ = pd.read_parquet(path, columns=["judul", "bulan"])
    c = collections.Counter()
    for t, b in zip(df_["judul"], df_["bulan"]):
        for kata in tokenisasi(t):
            if kata not in STOPWORD_DOMAIN:
                c[(b, kata)] += 1
    return c

gabung_bulan = collections.Counter()
for p in splits:
    gabung_bulan.update(mapper_bulan(p))
top_bulan = top_per_kategori(gabung_bulan, n=8)
for b, daftar in top_bulan.items():
    print(f"{b}: ", ", ".join(f"{k} ({n:,})" for k, n in daftar))

jan = {k for k, _ in top_bulan["2020-01"]}; mei = {k for k, _ in top_bulan["2020-05"]}
print("\nHanya di 8 teratas Januari:", sorted(jan - mei))
print("Hanya di 8 teratas Mei     :", sorted(mei - jan))

2020-01:  banjir (1,220), polisi (554), jokowi (516), tahun (493), baru (475), mobil (428), minta (427), kpk (415)
2020-02:  corona (1,076), polisi (630), virus (596), banjir (572), anak (453), indonesia (415), jakarta (407), wni (404)
2020-03:  corona (6,986), virus (1,340), positif (925), pasien (749), rumah (695), cegah (665), indonesia (585), jokowi (537)
2020-04:  corona (4,914), positif (828), pandemi (706), psbb (689), warga (609), meninggal (577), rumah (576), kasus (557)
2020-05:  corona (2,935), kasus (640), positif (628), pandemi (596), warga (565), lebaran (562), psbb (522), covid (522)
2020-06:  corona (943), normal (471), new (465), kasus (315), juni (262), polisi (246), baru (225), psbb (214)

Hanya di 8 teratas Januari: ['banjir', 'baru', 'jokowi', 'kpk', 'minta', 'mobil', 'polisi', 'tahun']
Hanya di 8 teratas Mei     : ['corona', 'covid', 'kasus', 'lebaran', 'pandemi', 'positif', 'psbb', 'warga']


### Latihan 3 — Bigram

In [31]:
def mapper_bigram(path):
    judul = pd.read_parquet(path, columns=["judul"])["judul"]
    out = []
    for t in judul:
        tok = tokenisasi(t)
        out.extend((f"{a} {b}", 1) for a, b in zip(tok, tok[1:]))
    return out

def mapper_bigram_combiner(path):
    return collections.Counter(k for k, _ in mapper_bigram(path))

bg_map = [mapper_bigram(p) for p in splits]
bg_comb = [mapper_bigram_combiner(p) for p in splits]
bg_total = collections.Counter()
for c in bg_comb:
    bg_total.update(c)
print("Pasangan bigram tanpa combiner:", f"{sum(len(h) for h in bg_map):,}")
print("Pasangan bigram dengan combiner:", f"{sum(len(c) for c in bg_comb):,}")
print("Bigram unik                    :", f"{len(bg_total):,}")
print("Frekuensi 'virus corona'       :", f"{bg_total['virus corona']:,}")
print("\n10 bigram teratas:")
for k, n in bg_total.most_common(10):
    print(f"  {k:<20} {n:>6,}")

Pasangan bigram tanpa combiner: 734,783
Pasangan bigram dengan combiner: 411,217
Bigram unik                    : 349,114
Frekuensi 'virus corona'       : 2,915

10 bigram teratas:
  virus corona          2,915
  corona di             2,178
  positif corona        1,804
  di tengah             1,258
  pandemi corona        1,075
  di rumah              1,047
  new normal              917
  di indonesia            871
  rp juta                 700
  hari ini                627


### Latihan 4 — Jumlah reducer R = 2, 8, 32

In [32]:
tabel_reducer = []
for r in (2, 4, 8, 16, 32, 64):
    b = beban(hasil_map, r)
    tabel_reducer.append({"reducer": r, "terberat_%": round(max(b)/sum(b)*100, 2),
                          "bagian_adil_%": round(100/r, 2), "rasio": round(max(b)/sum(b) / (1/r), 2)})
tabel_reducer = pd.DataFrame(tabel_reducer)
print(tabel_reducer.to_string(index=False))

 reducer  terberat_%  bagian_adil_%  rasio
       2       52.27          50.00   1.05
       4       29.17          25.00   1.17
       8       16.83          12.50   1.35
      16        8.82           6.25   1.41
      32        5.82           3.12   1.86
      64        4.28           1.56   2.74


### Latihan 5 — Crawl ulang dengan jeda

In [33]:
t0 = time.perf_counter()
n_judul = sum(1 for _ in crawl_iter(BASE, range(1, 51), delay=0.05))
w50 = time.perf_counter() - t0
per_hal = w50 / 50
print(f"50 halaman, jeda 0,05 s: {w50:.1f} detik ({n_judul:,} judul) -> {per_hal:.3f} s/halaman")
est = N_HALAMAN * (1.0 + (per_hal - 0.05))
print(f"Perkiraan 911 halaman dengan jeda 1 detik: {est:.0f} detik ≈ {est/60:.1f} menit")

50 halaman, jeda 0,05 s: 4.7 detik (5,000 judul) -> 0.094 s/halaman
Perkiraan 911 halaman dengan jeda 1 detik: 951 detik ≈ 15.9 menit


### Latihan 6 — Sitemap lain (opsional, butuh internet)
Ganti satu alamat dengan `sitemap_news.xml` dari bagian lain yang tercantum di `https://www.detik.com/sitemap.xml`, lalu terapkan `aturan_kualitas`. Jalankan **hanya sekali**. Biarkan `SITEMAP_LAIN = None` bila tidak dikerjakan.

In [34]:
SITEMAP_LAIN = "https://finance.detik.com/berita-ekonomi-bisnis/sitemap_news.xml"

if SITEMAP_LAIN:
    try:
        sampel2 = crawl_detik([SITEMAP_DIPILIH[0], SITEMAP_DIPILIH[1], SITEMAP_LAIN])
        d2 = pd.DataFrame(sampel2)
        print("Kategori:", d2["kategori"].value_counts().to_dict())
        for nama, lulus in aturan_kualitas(d2).items():
            print(f"{nama:<24} gagal: {(~lulus).sum():>3}")
    except Exception as e:
        print("Gagal:", type(e).__name__, e)
else:
    print("Latihan 6 dilewati.")

Kategori: {'news': 7, 'inet': 7, 'finance': 6}
judul tidak kosong       gagal:   0
judul 3-30 kata          gagal:   0
kategori dikenal         gagal:   0
tanggal valid            gagal:   0
url artikel valid        gagal:   0
judul tidak duplikat     gagal:   0


### Latihan 7 — Kata kunci terbanyak (MapReduce pada hasil crawl H-4)

In [35]:
if sampel:
    def mapper_kunci(entri):
        return [(k.strip().lower(), 1) for k in entri["kata_kunci"].split(",") if k.strip()]
    shuf = collections.defaultdict(list)
    for e in sampel:
        for k, v in mapper_kunci(e):
            shuf[k].append(v)
    hasil_kunci = collections.Counter({k: sum(v) for k, v in shuf.items()})
    print("Entri dengan kata kunci:", sum(1 for e in sampel if e["kata_kunci"]), "dari", len(sampel))
    print("Kata kunci teratas:", hasil_kunci.most_common(10))
else:
    print("Tidak ada hasil crawl H-4 (crawl gagal / belum dijalankan).")

Entri dengan kata kunci: 20 dari 20
Kata kunci teratas: [('tentara bayaran', 3), ('jabodetabek', 2), ('wni', 2), ('rusia', 2), ('registrasi sim card biometrik', 2), ('komdigi', 2), ('basket', 2), ('ibl all indonesian 2026', 2), ('ibl all indonesian', 2), ('pengedar obat keras', 1)]


# O. Tugas Mandiri — Inverted Index dengan MapReduce
Mapper menghasilkan `(kata, id)`, shuffle mengelompokkan per kata, reducer menggabungkan id menjadi daftar terurut tanpa duplikat.

**Penanganan id kosong (kategori inet, 5.577 judul):** baris *tidak* dibuang. Dibuat **id pengganti** `inet-<crc32 judul>` yang (1) deterministik (sama di setiap komputer/sesi, beda dengan nomor baris yang bergantung urutan), (2) jelas dibedakan dari id asli detik sehingga tidak dikira id artikel sungguhan, dan (3) unik karena judul sudah dideduplikasi di H-6 (dicek di bawah).

In [36]:
def id_dokumen(df_):
    asli = df_["id"].astype("Int64").astype(str)
    pengganti = "inet-" + df_["judul"].map(lambda j: f"{zlib.crc32(j.encode()):08x}")
    return asli.where(df_["id"].notna(), pengganti)

# Cek keunikan id pengganti
ids_semua = id_dokumen(bersih)
print("Total dokumen:", f"{len(ids_semua):,}", "| id unik:", f"{ids_semua.nunique():,}",
      "| id pengganti:", f"{ids_semua.str.startswith('inet-').sum():,}")

def mapper_index(path):
    df_ = pd.read_parquet(path, columns=["judul", "id"])
    keluaran = []
    for doc, t in zip(id_dokumen(df_), df_["judul"]):
        for kata in set(tokenisasi(t)):          # set: satu pasangan per kata per judul
            keluaran.append((kata, doc))
    return keluaran

def kunci_urut(doc):                              # id asli (angka) dulu, lalu id pengganti
    return (1, doc) if doc.startswith("inet-") else (0, int(doc))

def reducer_index(kata, daftar_id):
    return kata, sorted(set(daftar_id), key=kunci_urut)

pasangan_index = [mapper_index(p) for p in splits]
shuf_idx = collections.defaultdict(list)
for h in pasangan_index:
    for kata, doc in h:
        shuf_idx[kata].append(doc)
inverted = dict(reducer_index(k, v) for k, v in shuf_idx.items())

print("Pasangan (kata, id) dari mapper:", f"{sum(len(h) for h in pasangan_index):,}")
print("Kata dalam indeks              :", f"{len(inverted):,}")

# Validasi: panjang posting list = jumlah judul yang memuat kata itu
uji = "corona"
manual = bersih["judul"].map(lambda t: uji in set(tokenisasi(t))).sum()
print(f"Validasi '{uji}': posting list {len(inverted[uji]):,} = judul memuat kata {manual:,} ->", len(inverted[uji]) == manual)

CONTOH_KATA = ["lockdown", "mudik", "vaksin"]
print()
for k in CONTOH_KATA:
    ids = inverted.get(k, [])
    print(f"{k:<9} muncul di {len(ids):>5,} judul | 5 id pertama: {ids[:5]}")

Total dokumen: 90,818 | id unik: 90,818 | id pengganti: 5,577
Pasangan (kata, id) dari mapper: 817,993
Kata dalam indeks              : 32,275
Validasi 'corona': posting list 17,228 = judul memuat kata 17,228 -> True

lockdown  muncul di   546 judul | 5 id pertama: ['4880125', '4889884', '4910645', '4910883', '4922500']
mudik     muncul di   888 judul | 5 id pertama: ['4852294', '4853938', '4856814', '4856929', '4857196']
vaksin    muncul di   188 judul | 5 id pertama: ['4862824', '4873864', '4874997', '4875084', '4875552']


In [37]:
# Tunjukkan judul asli dari id pertama tiap kata contoh (bukti indeks benar)
peta_judul = dict(zip(ids_semua, bersih["judul"]))
for k in CONTOH_KATA:
    if inverted.get(k):
        print(f"{k:<9} -> {inverted[k][0]}: {peta_judul[inverted[k][0]]}")

# Simpan ke lapisan siap_pakai (Parquet + database laporan)
idx_df = pd.DataFrame({"kata": list(inverted), "jumlah_dokumen": [len(v) for v in inverted.values()],
                       "daftar_id": [",".join(v) for v in inverted.values()]})
idx_df.to_parquet(SIAP / "inverted_index.parquet", index=False)
db = sqlite3.connect(SIAP / "laporan.db")
idx_df.to_sql("inverted_index", db, if_exists="replace", index=False)
print("\nBaris inverted_index diekspor:", db.execute("SELECT COUNT(*) FROM inverted_index").fetchone()[0])
db.close()

lockdown  -> 4880125: Kemlu soal Evakuasi WNI: Status Wuhan China Masih Lockdown
mudik     -> 4852294: Selama Mudik Nataru, Konsumsi BBM Naik 16% hingga 27,9 KL/Hari
vaksin    -> 4862824: Heboh Pneumonia 'Misterius' China, Perlukah Minta Vaksin?

Baris inverted_index diekspor: 32275


In [38]:
def aman(x):
    try: return x()
    except Exception as e: return f"ERROR: {e}"

ringkasan = {
    "H2_csv": list(df_sumber.shape),
    "H4_crawl_detik": aman(lambda: {"jumlah": len(sampel),
                                    "kategori": pd.DataFrame(sampel)["kategori"].value_counts().to_dict() if sampel else {},
                                    "contoh": [{k: x[k] for k in ("judul", "kategori", "tanggal")} for x in sampel[:5]]}),
    "H5": {**statistik, "waktu_crawl_s": round(waktu_crawl, 1),
           "ukuran_mentah_MB": round(sum(p.stat().st_size for p in RAW.glob('*.jsonl')) / 1e6, 1)},
    "H6_kualitas_gagal": {k: int((~v).sum()) for k, v in aturan_kualitas(mentah).items()},
    "H6_bersih": len(bersih), "H6_partisi": hasil_partisi,
    "H6_pruning": [len(hanya), len(semua)],
    "H7_per_kategori": dict(zip(per_kat_df.kategori, per_kat_df.jumlah.astype(int))),
    "H7_corona_bulan": dict(zip(corona_df.bulan, corona_df.judul_corona.astype(int))),
    "H7_corona_kategori": corona_kat_df.to_dict("records"),
    "H9": {"pasangan": total_pasangan, "kata_unik": len(hasil_semua), "top10_tanpa_stopword": top[:10],
           "peringkat_rp_ri_the": {k: peringkat.get(k) for k in ("rp", "ri", "the")}},
    "H10_cpu": os.cpu_count(), "H10": tabel_h10,
    "H11_4reducer_tanpa_combiner": beban(hasil_map), "H11_4reducer_dengan_combiner": beban([list(c.items()) for c in h2]),
    "H11_skew": tabel_reducer.to_dict("records"),
    "H12_pipe_sama": hasil_pipe == hasil_semua,
    "H12_tanpa_sort_baris": sum(len(v) for v in tanpa_sort.values()),
    "H13_top5": top5_awal, "L1_top5_domain": top5_domain,
    "L2_top_bulan": top_bulan,
    "L3_bigram": {"tanpa_combiner": sum(len(h) for h in bg_map), "dengan_combiner": sum(len(c) for c in bg_comb),
                  "top10": bg_total.most_common(10)},
    "L5": {"w50_s": round(w50, 1), "est_911_jeda1_s": round(est)},
    "O_inverted": {"kata": len(inverted), "contoh": {k: inverted.get(k, [])[:5] for k in CONTOH_KATA},
                   "jumlah_dokumen": {k: len(inverted.get(k, [])) for k in CONTOH_KATA}},
}
Path("ringkasan_laporan.json").write_text(json.dumps(ringkasan, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(ringkasan, ensure_ascii=False, indent=1, default=str)[:3000], "...")

{
 "H2_csv": [
  91017,
  4
 ],
 "H4_crawl_detik": {
  "jumlah": 20,
  "kategori": {
   "news": 7,
   "inet": 7,
   "sport": 6
  },
  "contoh": [
   {
    "judul": "Polisi Tangkap Pengedar Obat Keras di Bogor, 282.500 Butir Tramadol Disita",
    "kategori": "news",
    "tanggal": "2026-11-13"
   },
   {
    "judul": "WNI Tentara Bayaran Rusia Dijanjikan Bonus Jika Bisa Tembak Tentara Ukraina",
    "kategori": "news",
    "tanggal": "2026-11-13"
   },
   {
    "judul": "Di COP31 Turki, DPD RI Tawarkan 3 Usulan Kebijakan Komitmen Iklim",
    "kategori": "news",
    "tanggal": "2026-11-13"
   },
   {
    "judul": "5 Orang Sindikat Pencuri Perhiasan Lansia Modus Terapis di Depok Ditangkap",
    "kategori": "news",
    "tanggal": "2026-11-13"
   },
   {
    "judul": "15 WNI Jadi Tentara Bayaran Rusia, Rata-rata Mantan Admin Judol",
    "kategori": "news",
    "tanggal": "2026-11-13"
   }
  ]
 },
 "H5": {
  "max_antrean": 200,
  "masuk": 91017,
  "keluar": 91017,
  "file": 19,
  "waktu_crawl

# Pengumpulan - zip folder lapisan

In [39]:
import zipfile
NAMA_ZIP = "BD_P05_2411532011_AufanTaufiqurrahman_lapisan.zip"
with zipfile.ZipFile(NAMA_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(RAW / "batch_0001.jsonl", "lapisan_mentah/batch_0001.jsonl")
    for p in Path(TERKURASI).rglob("*.parquet"):
        z.write(p, str(p))
    for p in SIAP.iterdir():
        z.write(p, str(p))
    if Path("contoh_crawl/detik_sitemap.jsonl").exists():
        z.write("contoh_crawl/detik_sitemap.jsonl")
print(NAMA_ZIP, round(os.path.getsize(NAMA_ZIP) / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(NAMA_ZIP); files.download("ringkasan_laporan.json")
except ImportError:
    pass

BD_P05_2411532011_AufanTaufiqurrahman_lapisan.zip 15.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>